# 05. BitNet b1.58: Native Ternary Transformers

- **Category:** 14_quantisation / 04_vector_lattice_qat_and_lowbit
- **Lineage & Citations:** Wang et al. (2024, *The Era of 1-bit LLMs: All Large Language Models are in 1.58 Bits*, Microsoft Research); Ma et al. (2024, *The Era of 1-bit LLMs: Training Tips, Code and FAQ*).
- **Purpose:** Implement BitLinear native ternary transformer layers (weights $\in \{-1, 0, +1\}$, activations in 8-bit integers) that eliminate floating-point multiplications from core GEMMs while matching 16-bit LLaMA perplexity.
- **Topic Connections:** `🔗 Upstream Prerequisites:` [04. Binary Sub1bit and Ternary BNN XNOR BiLLM OneBit TWN TTQ](04_binary_sub1bit_and_ternary_bnn_xnor_billm_onebit_twn_ttq.ipynb) $\to$ `🔗 Downstream Syntheses:` [06. QLoRA LoftQ and QA-LoRA Quantized PEFT](06_qlora_loftq_and_qa_lora_quantized_peft.ipynb)

> [!WARNING]
> **Decision Boundary: When NOT to use it**  
> BitNet b1.58 requires native training **from scratch** with Straight-Through Estimators (STE); it is NOT a post-training quantization (PTQ) method for existing FP16 checkpoints without full continued pretraining. Do NOT omit RMSNorm (SubLN) prior to activation quantization (without SubLN, activation outliers destabilize integer dynamic range).

## Mathematical Foundations & Governing Equations

1. **RMSNorm (SubLN)**:
   $$\mathbf{X}_{\text{norm}} = \frac{\mathbf{X}}{\sqrt{\frac{1}{d_{\text{in}}} \sum_{j=1}^{d_{\text{in}}} X_{ij}^2 + \epsilon}} \odot \mathbf{w}_{\text{rms}}$$
2. **Per-Token Dynamic INT8 Activation Quantization**:
   $$\gamma_i = \max_j |X_{\text{norm}, ij}|, \quad s_{X, i} = \max\left(\frac{\gamma_i}{127.0}, \epsilon\right), \quad \widetilde{\mathbf{X}}_i = \text{clamp}\left(\left\lfloor \frac{\mathbf{X}_{\text{norm}, i}}{s_{X, i}} \right\rceil, -127, 127\right)$$
3. **Tensor-Wide Ternary Weight Quantization**:
   $$\beta = \frac{1}{d_{\text{in}} d_{\text{out}}} \sum_{i, j} |W_{ij}|, \quad s_W = \max(\beta, \epsilon), \quad \widetilde{\mathbf{W}} = \text{clamp}\left(\left\lfloor \frac{\mathbf{W}}{s_W} \right\rceil, -1, +1\right)$$
4. **Multiplication-Free Integer GEMM & Rescaling**:
   $$\mathbf{Y} = (\widetilde{\mathbf{X}} \cdot \widetilde{\mathbf{W}}) \odot \left( \frac{\boldsymbol{\gamma} \cdot \beta}{127.0} \right) + \mathbf{b}$$
   where $\widetilde{\mathbf{X}} \cdot \widetilde{\mathbf{W}}$ is performed purely via signed additions and subtractions.

In [ ]:
import sys, time
from pathlib import Path
import torch
from safetensors.torch import load_file

torch.manual_seed(42)
r = Path.cwd()
while not (r / "Cargo.toml").exists() and r != r.parent: r = r.parent
WEIGHTS_PATH = r / "weights/14_quantisation/04_vector_lattice_qat_and_lowbit/05_bitnet_b1_58_native_ternary_transformers.safetensors"
weights = load_file(str(WEIGHTS_PATH))
print(f"Loaded {len(weights)} tensors from {WEIGHTS_PATH.name}")

In [ ]:
def rms_norm(x: torch.Tensor, weight: torch.Tensor, eps: float = 1e-5) -> torch.Tensor:
    variance = torch.mean(x ** 2, dim=-1, keepdim=True)
    return x * torch.rsqrt(variance + eps) * weight

def activation_quant_bitnet(x: torch.Tensor, q_b: float = 127.0, eps: float = 1e-5):
    gamma = torch.max(torch.abs(x), dim=1, keepdim=True).values
    scale = torch.clamp(gamma / q_b, min=eps)
    x_int8 = torch.clamp(torch.round(x / scale), -q_b, q_b)
    return x_int8 * scale, x_int8, gamma

def weight_quant_bitnet(w: torch.Tensor, eps: float = 1e-5):
    beta = torch.mean(torch.abs(w), dim=(0, 1), keepdim=True)
    scale = torch.clamp(beta, min=eps)
    w_ternary = torch.clamp(torch.round(w / scale), -1.0, 1.0)
    return w_ternary * scale, w_ternary, beta

def bitlinear_forward(x: torch.Tensor, weight: torch.Tensor, rms_weight: torch.Tensor, bias: torch.Tensor | None = None):
    x_norm = rms_norm(x, rms_weight)
    x_q, _, _ = activation_quant_bitnet(x_norm, q_b=127.0)
    w_q, _, _ = weight_quant_bitnet(weight)
    y = x_q @ w_q
    return (y + bias) if bias is not None else y

In [ ]:
# 1. Linear anchor parity
y_anchor = weights["test_input"] @ weights["weight"] + weights["bias"]
assert float(torch.max(torch.abs(y_anchor - weights["test_output"]))) < 1e-5

# 2. RMSNorm and quantization operators
x_norm = rms_norm(weights["x_raw"], weights["rms_weight"])
assert float(torch.max(torch.abs(x_norm - weights["x_norm"]))) < 1e-5

x_q, x_int8, gamma = activation_quant_bitnet(x_norm, 127.0)
assert float(torch.max(torch.abs(x_q - weights["x_q"]))) < 1e-5
assert float(torch.max(torch.abs(x_int8 - weights["x_int8"]))) < 1e-5
assert float(torch.max(torch.abs(gamma - weights["gamma"]))) < 1e-5

w_q, w_ternary, beta = weight_quant_bitnet(weights["w_orig"])
assert float(torch.max(torch.abs(w_q - weights["w_q"]))) < 1e-5
assert float(torch.max(torch.abs(w_ternary - weights["w_ternary"]))) < 1e-5
assert float(torch.max(torch.abs(beta - weights["beta"]))) < 1e-5

# 3. BitLinear end-to-end forward pass
out_bl = bitlinear_forward(weights["x_raw"], weights["w_orig"], weights["rms_weight"], weights["b_orig"])
assert float(torch.max(torch.abs(out_bl - weights["out_bitlinear"]))) < 1e-5
print("All BitNet b1.58 parity assertions verified against golden checkpoint!")

In [ ]:
# Multiplier-Free Integer Accumulation vs FP32 Baseline SQNR
int_acc = weights["x_int8"] @ weights["w_ternary"]
assert float(torch.max(torch.abs(int_acc - weights["integer_acc"]))) < 1e-5

out_scaled = int_acc * (weights["gamma"] * weights["beta"] / 127.0) + weights["b_orig"]
assert float(torch.max(torch.abs(out_scaled - weights["out_integer_scaled"]))) < 1e-5

y_fp32 = weights["out_fp32"]
mse = float(torch.mean((y_fp32 - out_scaled) ** 2))
sig_pwr = float(torch.mean(y_fp32 ** 2))
sqnr = 10.0 * torch.log10(torch.tensor(sig_pwr / (mse + 1e-12))).item()

zero_ratio = float(torch.mean((weights["w_ternary"] == 0.0).float())) * 100.0
print(f"Ternary Weight Zero Fraction: {zero_ratio:.1f}%")
print(f"BitNet b1.58 Output SQNR vs FP32: {sqnr:.2f} dB")
assert sqnr > 4.0, "BitNet b1.58 must maintain viable output fidelity (>4 dB SQNR)!"
assert zero_ratio > 10.0, "Native ternary weights must include non-trivial zero-state sparsity!"

In [ ]:
# Latency Microbenchmark: Multiplication-Free Matrix Accumulation (1024x1024)
b, d_in, d_out = 16, 1024, 1024
x_int_bench = torch.randint(-127, 128, (b, d_in)).float()
w_tern_bench = torch.randint(-1, 2, (d_in, d_out)).float()

t0 = time.perf_counter()
for _ in range(50):
    res = x_int_bench @ w_tern_bench
t_matmul = (time.perf_counter() - t0) / 50 * 1e3

print(f"Integer-Ternary MatMul ({b}x{d_in} @ {d_in}x{d_out}) Latency: {t_matmul:.3f} ms")
assert t_matmul < 25.0, "Integer matrix accumulation must execute in <25 ms on CPU"

## Algorithmic Takeaways & Cross-Domain Isomorphisms

### Key Takeaways
1. **Zero Multiplications in GEMMs**: Weight ternary states $\{-1, 0, +1\}$ convert floating-point multiplications into signed integer additions and subtractions.
2. **SubLN Dynamic Range Control**: RMSNorm directly before activation quantization prevents activation outliers from saturating the 8-bit dynamic range.
3. **Extreme Energy and Memory Scaling**: BitNet b1.58 reduces matrix multiplication energy by up to $71.4\times$ on custom accelerators, enabling 70B models to run comfortably on a single GPU.

---
### Synchronized Track Navigation
- **Architectural Reference (Markdown):** [05_bitnet_b1_58_native_ternary_transformers.md](../../../markdown/14_quantisation/04_vector_lattice_qat_and_lowbit/05_bitnet_b1_58_native_ternary_transformers.md)
- **PyTorch 2.x Module:** [05_bitnet_b1_58_native_ternary_transformers.py](../../../pytorch2/14_quantisation/04_vector_lattice_qat_and_lowbit/05_bitnet_b1_58_native_ternary_transformers.py)
- **Functional JAX Module:** [05_bitnet_b1_58_native_ternary_transformers.py](../../../jax/14_quantisation/04_vector_lattice_qat_and_lowbit/05_bitnet_b1_58_native_ternary_transformers.py)
- **Native Rust Candle Track:** [05_bitnet_b1_58_native_ternary_transformers.rs](../../../candle/14_quantisation/04_vector_lattice_qat_and_lowbit/05_bitnet_b1_58_native_ternary_transformers.rs)
- **Hardware Kernel (CPU / WGSL):** [05_bitnet_b1_58_native_ternary_transformers.rs](../../../kernels/14_quantisation/04_vector_lattice_qat_and_lowbit/05_bitnet_b1_58_native_ternary_transformers.rs)
- **Golden SafeTensors:** [05_bitnet_b1_58_native_ternary_transformers.safetensors](../../../weights/14_quantisation/04_vector_lattice_qat_and_lowbit/05_bitnet_b1_58_native_ternary_transformers.safetensors)

---
**Curriculum Navigation:**  
&larr; [Previous: 04. Binary Sub1bit and Ternary BNN XNOR BiLLM OneBit TWN TTQ](04_binary_sub1bit_and_ternary_bnn_xnor_billm_onebit_twn_ttq.ipynb) | [Master Index](../../../README.md#14_quantisation) | [Next: 06. QLoRA LoftQ and QA-LoRA Quantized PEFT](06_qlora_loftq_and_qa_lora_quantized_peft.ipynb) &rarr;